In [1]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Define data transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),    # Resize images to 224x224
    transforms.ToTensor(),            # Convert images to tensor
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # Normalize images
])

# Load datasets
train_dataset = datasets.ImageFolder(root='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train', transform=transform)
val_dataset = datasets.ImageFolder(root='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid', transform=transform)

# DataLoader with batch size of 64 and 4 worker processes
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4)

class CustomCNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=5, padding=2)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # Calculate the output size after convolution and pooling
        # Assuming input image size is (224, 224)
        self.fc1 = nn.Linear(512 * 7 * 7, 128)  # Adjust the size based on the image dimensions
        self.fc2 = nn.Linear(128, 64)
        self.fc_out = nn.Linear(64, num_classes)  # Output layer for classification
        self.activation = nn.ReLU()

    def forward(self, x):
        # Forward pass through the network
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = self.pool(self.activation(self.conv3(x)))
        x = self.pool(self.activation(self.conv4(x)))
        x = self.pool(self.activation(self.conv5(x)))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc_out(x)
        return x

# Initialize the model
num_classes = len(train_dataset.classes)
model = CustomCNNModel(num_classes=num_classes)

# Move model to the appropriate device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)  # AdamW optimizer with weight decay

# Mixed precision training setup
scaler = torch.cuda.amp.GradScaler()

# Early stopping parameters
patience = 7
best_loss = float('inf')
epochs_without_improvement = 0

num_epochs = 40  # Number of epochs to train

# Record the start time
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    epoch_start_time = time.time()  # Record the start time for the epoch
    
    # Training phase
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():  # Mixed precision
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
    
    train_loss = running_loss / len(train_dataset)
    
    # Validation phase
    model.eval()
    val_running_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():  # Mixed precision
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    val_loss = val_running_loss / len(val_dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    
    epoch_time = time.time() - epoch_start_time  # Calculate epoch duration
    print(f'Epoch {epoch + 1}/{num_epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {accuracy:.4f}, Time: {epoch_time:.2f} seconds')
    
    # Check for early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        epochs_without_improvement = 0
        # Save the best model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping at epoch {epoch + 1}')
            break

# Record the total training time
total_time = time.time() - start_time
print(f'Total Training Time: {total_time:.2f} seconds')


Epoch 1/40, Training Loss: 1.5165, Validation Loss: 0.6138, Validation Accuracy: 0.8059, Time: 145.17 seconds


Epoch 2/40, Training Loss: 0.4680, Validation Loss: 0.3400, Validation Accuracy: 0.8917, Time: 141.44 seconds


Epoch 3/40, Training Loss: 0.2451, Validation Loss: 0.2429, Validation Accuracy: 0.9200, Time: 142.57 seconds


Epoch 4/40, Training Loss: 0.1568, Validation Loss: 0.1921, Validation Accuracy: 0.9373, Time: 140.14 seconds


Epoch 5/40, Training Loss: 0.1054, Validation Loss: 0.1614, Validation Accuracy: 0.9461, Time: 143.19 seconds


...

Epoch 15/40, Training Loss: 0.0327, Validation Loss: 0.1480, Validation Accuracy: 0.9641, Time: 140.98 seconds


Epoch 16/40, Training Loss: 0.0389, Validation Loss: 0.1278, Validation Accuracy: 0.9676, Time: 139.41 seconds


Epoch 17/40, Training Loss: 0.0306, Validation Loss: 0.1618, Validation Accuracy: 0.9605, Time: 141.72 seconds


Epoch 18/40, Training Loss: 0.0231, Validation Loss: 0.1694, Validation Accuracy: 0.9587, Time: 138.15 seconds


Epoch 19/40, Training Loss: 0.0327, Validation Loss: 0.1868, Validation Accuracy: 0.9540, Time: 140.56 seconds
Early stopping at epoch 19
Total Training Time: 2683.71 seconds
